# 4. AML Monitoring — Ongoing Surveillance

**Goal**: Build a monitoring dashboard that:
1. Tracks flagged wallets for new activity
2. Detects suspicious patterns (large transfers, mixer usage, bridge hops)
3. Generates alerts when thresholds are exceeded
4. Provides drift detection for classification model

**Integration**: Uses existing `MonitoredWallet` + `Alert` tables + ML classification pipeline.

**Stack**: Plotly dashboards, Evidently drift reports, Celery scheduled tasks.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta, timezone

from notebooks.src.data_loader import DataLoader
from notebooks.src.investigation_visualizer import InvestigationVisualizer as iviz

loader = DataLoader()
print('Connected')

## 4.1 Active Investigations Overview

In [ ]:
# Load investigations and their current state
investigations = loader.run_sql("""
    SELECT i.id, i.name, i.status, i.reported_loss_usd,
           COUNT(DISTINCT iw.address) as wallet_count,
           COUNT(DISTINCT it.id) as transfer_count,
           SUM(CASE WHEN iw.is_flagged THEN 1 ELSE 0 END) as flagged_wallets
    FROM investigation i
    LEFT JOIN investigation_wallet iw ON i.id = iw.investigation_id
    LEFT JOIN investigation_transfer it ON i.id = it.investigation_id
    GROUP BY i.id, i.name, i.status, i.reported_loss_usd
    ORDER BY i.id
""")

print(f'Total investigations: {len(investigations)}')
display(investigations)

## 4.2 Alert History

In [ ]:
# Load recent alerts from monitored wallets
alerts = loader.run_sql("""
    SELECT a.id, a.alert_type, a.amount, a.token, a.risk_score,
           a.counterparty, a.timestamp, a.is_read,
           mw.address, mw.chain_code, mw.label
    FROM alert a
    JOIN monitored_wallet mw ON a.wallet_id = mw.id
    ORDER BY a.timestamp DESC
    LIMIT 100
""")

if not alerts.empty:
    print(f'Recent alerts: {len(alerts)}')
    
    # Alert type distribution
    fig = go.Figure(go.Pie(
        labels=alerts['alert_type'].value_counts().index,
        values=alerts['alert_type'].value_counts().values,
        hole=0.4,
    ))
    fig.update_layout(title='Alert Types Distribution', template='plotly_white', height=350)
    fig.show()
    
    # Risk score distribution
    fig = go.Figure(go.Histogram(x=alerts['risk_score'], nbinsx=20, marker_color='#d62728'))
    fig.update_layout(title='Alert Risk Score Distribution', template='plotly_white', height=350)
    fig.show()
else:
    print('No alerts yet. Set up monitoring first.')

## 4.3 Model Drift Detection

Compare the feature distribution of recent classifications vs training data.

In [ ]:
# Load wallet scores for drift analysis
scores = loader.get_wallet_scores()

if not scores.empty and len(scores) > 20:
    feature_cols = ['tx_count', 'unique_counterparties', 'avg_tx_value', 'max_tx_value', 'in_out_ratio']
    available_cols = [c for c in feature_cols if c in scores.columns]
    
    if available_cols:
        fig = make_subplots(
            rows=len(available_cols), cols=1,
            subplot_titles=available_cols,
            vertical_spacing=0.06,
        )
        
        for i, col in enumerate(available_cols, 1):
            data = scores[col].dropna()
            fig.add_trace(go.Histogram(
                x=data, nbinsx=30, name=col,
                marker_color='#1f77b4', opacity=0.7,
            ), row=i, col=1)
        
        fig.update_layout(
            title='Feature Distributions (Current Data)',
            template='plotly_white',
            height=200 * len(available_cols),
            showlegend=False,
        )
        fig.show()
        
        # Basic drift stats
        print('\nFeature Statistics:')
        display(scores[available_cols].describe())
    else:
        print('No feature columns available in scores data')
else:
    print('Not enough scored wallets for drift analysis.')
    print('Run classify_investigation_wallets on an investigation first.')

## 4.4 Monitoring Rules Engine

Define alert rules for ongoing monitoring.

In [ ]:
# Define monitoring rules
MONITORING_RULES = {
    'large_transfer': {
        'description': 'Single transfer > $50,000',
        'threshold_usd': 50_000,
        'risk_score': 0.7,
        'alert_type': 'large_transfer',
    },
    'mixer_usage': {
        'description': 'Any transfer to a known mixer',
        'risk_score': 0.9,
        'alert_type': 'mixer',
    },
    'bridge_hop': {
        'description': 'Transfer to a known bridge',
        'risk_score': 0.6,
        'alert_type': 'bridge',
    },
    'rapid_distribution': {
        'description': '10+ outgoing transfers in 1 hour',
        'threshold_count': 10,
        'window_hours': 1,
        'risk_score': 0.8,
        'alert_type': 'outgoing',
    },
    'dormant_reactivation': {
        'description': 'Wallet inactive >30 days suddenly active',
        'dormancy_days': 30,
        'risk_score': 0.5,
        'alert_type': 'new_activity',
    },
}

print('Monitoring Rules:')
for name, rule in MONITORING_RULES.items():
    print(f"  [{rule['alert_type'].upper()}] {name}: {rule['description']} (risk: {rule['risk_score']})")

## 4.5 Simulate Alert Engine

In [ ]:
# Simulate applying monitoring rules to investigation transfers
INVESTIGATION_ID = 1

transfers = loader.get_transfers(INVESTIGATION_ID)
known_entities = loader.get_known_entities()

if not transfers.empty:
    mixer_addrs = set(known_entities[known_entities['type'] == 'mixer']['address'].str.lower()) if not known_entities.empty else set()
    bridge_addrs = set(known_entities[known_entities['type'] == 'bridge']['address'].str.lower()) if not known_entities.empty else set()
    
    simulated_alerts = []
    
    for _, tx in transfers.iterrows():
        to_addr = (tx.get('to_address') or '').lower()
        value = tx.get('value', 0) or 0
        
        # Large transfer rule
        if value > 50000:
            simulated_alerts.append({
                'rule': 'large_transfer', 'value': value,
                'from': tx.get('from_address', '')[:12],
                'to': to_addr[:12],
                'timestamp': tx.get('timestamp'),
                'risk': 0.7,
            })
        
        # Mixer usage
        if to_addr in mixer_addrs:
            simulated_alerts.append({
                'rule': 'mixer_usage', 'value': value,
                'from': tx.get('from_address', '')[:12],
                'to': to_addr[:12],
                'timestamp': tx.get('timestamp'),
                'risk': 0.9,
            })
        
        # Bridge hop
        if to_addr in bridge_addrs:
            simulated_alerts.append({
                'rule': 'bridge_hop', 'value': value,
                'from': tx.get('from_address', '')[:12],
                'to': to_addr[:12],
                'timestamp': tx.get('timestamp'),
                'risk': 0.6,
            })
    
    sim_df = pd.DataFrame(simulated_alerts)
    if not sim_df.empty:
        print(f'Simulated alerts: {len(sim_df)}')
        display(sim_df.groupby('rule').agg(
            count=('value', 'size'),
            total_value=('value', 'sum'),
            avg_risk=('risk', 'mean'),
        ))
    else:
        print('No alerts triggered — low risk investigation')
else:
    print('No transfer data available')

## 4.6 Entity Summary Dashboard

In [ ]:
# Summary: how much money ended up at which entity type
wallets = loader.get_wallets(INVESTIGATION_ID)

if not wallets.empty:
    # Sankey of fund flows
    edges_df = loader.get_edges(INVESTIGATION_ID)
    if not edges_df.empty:
        fig = iviz.plot_entity_sankey(edges_df, wallets, min_value=0.1)
        fig.show()
    
    # Role distribution
    fig = iviz.plot_role_distribution(wallets)
    fig.show()
    
    # Depth histogram
    fig = iviz.plot_depth_histogram(wallets)
    fig.show()
else:
    print('No wallet data')

## 4.7 Limitations & Next Steps

| Limitation | Impact | Mitigation |
|:-----------|:-------|:-----------|
| No real-time stream | Polling-based monitoring only | Celery beat scheduled every 5min |
| USD value approximation | No live price feeds | CoinGecko price backfill task |
| Single-model drift | Only one classification model tracked | Add Evidently scheduled reports |
| No Slack/email alerts | Alerts only visible in DB/UI | Add notification webhook |

**TAG: `needs_scheduled_monitoring`** — Set up Celery beat task for periodic wallet checks.  
**TAG: `aria_manual_escalation`** — Risk score > 0.9 should trigger manual review by Aria.